In [ ]:
import snowflake.connector
from neo4j import GraphDatabase
import pandas as pd
from pyvis.network import Network

# -----------------------------
# CONFIGURATION
# -----------------------------
SNOWFLAKE_CONFIG = {
    "user": "ASHISHCHOPRA",
    "password": "YOUR_SNOWFLAKE_PASSWORD",
    "account": "YZ31798",
    "warehouse": "AOGUIDE_WH",
    "database": "YOUR_DATABASE",
    "schema": "YOUR_SCHEMA"
}

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "YOUR_NEO4J_PASSWORD"

# -----------------------------
# STEP 1: FETCH DATA FROM SNOWFLAKE
# -----------------------------
def fetch_customers():
    try:
        conn = snowflake.connector.connect(**SNOWFLAKE_CONFIG)
        query = """
        SELECT CUSTOMER_ID, FULL_NAME, EMAIL, PHONE
        FROM CUSTOMERS
        WHERE FULL_NAME IS NOT NULL OR EMAIL IS NOT NULL OR PHONE IS NOT NULL
        """
        df = pd.read_sql(query, conn)
        conn.close()
        return df
    except Exception as e:
        raise RuntimeError(f"Error fetching data from Snowflake: {e}")

# -----------------------------
# STEP 2: LOAD DATA INTO NEO4J WITH WEIGHTS
# -----------------------------
def load_into_neo4j(df):
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        session.run("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Customer) REQUIRE c.id IS UNIQUE")

        # Insert nodes
        for _, row in df.iterrows():
            session.run("""
                MERGE (c:Customer {id: $id})
                SET c.name = $name, c.email = $email, c.phone = $phone
            """, id=row["CUSTOMER_ID"], name=row["FULL_NAME"], email=row["EMAIL"], phone=row["PHONE"])

        # Create weighted edges
        session.run("""
            MATCH (a:Customer), (b:Customer)
            WHERE a.id < b.id
            WITH a, b,
                 CASE WHEN a.name IS NOT NULL AND a.name = b.name THEN 1 ELSE 0 END +
                 CASE WHEN a.email IS NOT NULL AND a.email = b.email THEN 1 ELSE 0 END +
                 CASE WHEN a.phone IS NOT NULL AND a.phone = b.phone THEN 1 ELSE 0 END AS weight
            WHERE weight > 0
            MERGE (a)-[r:MATCHES]->(b)
            SET r.weight = weight
        """)
    driver.close()

# -----------------------------
# STEP 3: RUN GDS ANALYSIS (WEIGHTED)
# -----------------------------
def run_gds_analysis():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    with driver.session() as session:
        # Project weighted graph
        session.run("""
            CALL gds.graph.project(
                'customerGraph',
                'Customer',
                {MATCHES: {properties: 'weight'}}
            )
        """)

        # Weakly Connected Components
        result = session.run("""
            CALL gds.wcc.stream('customerGraph')
            YIELD nodeId, componentId
            RETURN gds.util.asNode(nodeId).id AS customerId, componentId
            ORDER BY componentId
        """)
        clusters = result.data()
    driver.close()
    return clusters

# -----------------------------
# STEP 4: NODE SIMILARITY ANALYSIS
# -----------------------------
def run_node_similarity(top_n=5, similarity_cutoff=0.5):
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    similarities = []
    with driver.session() as session:
        result = session.run(f"""
            CALL gds.nodeSimilarity.stream('customerGraph', {{ relationshipWeightProperty: 'weight' }})
            YIELD node1, node2, similarity
            WHERE similarity >= {similarity_cutoff}
            RETURN gds.util.asNode(node1).id AS customer1,
                   gds.util.asNode(node2).id AS customer2,
                   similarity
            ORDER BY similarity DESC
            LIMIT {top_n}
        """)
        similarities = result.data()
    driver.close()
    return similarities

# -----------------------------
# STEP 5: VISUALIZE WITH PYVIS (WEIGHTED)
# -----------------------------
def visualize_network():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    net = Network(height="750px", width="100%", bgcolor="#222222", font_color="white")

    with driver.session() as session:
        # Nodes
        nodes = session.run("""
            MATCH (c:Customer)
            RETURN c.id AS id, c.name AS name, c.email AS email, c.phone AS phone
        """)
        for record in nodes:
            label = record["name"] or record["email"] or record["phone"] or record["id"]
            net.add_node(record["id"], label=label, title=f"Email: {record['email']}<br>Phone: {record['phone']}")

        # Weighted edges
        edges = session.run("""
            MATCH (a:Customer)-[r:MATCHES]-(b:Customer)
            RETURN a.id AS source, b.id AS target, r.weight AS weight
        """)
        for record in edges:
            net.add_edge(
                record["source"],
                record["target"],
                value=record["weight"],  # Controls thickness
                title=f"Weight: {record['weight']}"
            )

    driver.close()
    net.show("customer_network.html")
    print("Weighted network visualization saved to customer_network.html")

# -----------------------------
# MAIN EXECUTION
# -----------------------------
if __name__ == "__main__":
    customers_df = fetch_customers()
    print(f"Fetched {len(customers_df)} customers from Snowflake.")

    load_into_neo4j(customers_df)
    print("Customer data loaded into Neo4j with weighted relationships.")

    clusters = run_gds_analysis()
    print(f"Found {len(set(c['componentId'] for c in clusters))} clusters.")

    similarities = run_node_similarity(top_n=10, similarity_cutoff=0.5)
    print("\nTop Similar Customers:")
    for sim in similarities:
        print(f"{sim['customer1']} ↔ {sim['customer2']} | similarity: {sim['similarity']:.2f}")

    visualize_network()
